In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

OPEN_API_KEY = os.getenv("OPEN_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
LAMGCHAIN_TRACING_V2 = os.getenv("LANGCHAIN_TRAICING_V2")


extaracting the data , extracting the elements of the pdf that will be use in the retrievel process

In [ ]:
from unstructured.partition.pdf import partition_pdf

file_path = "attention.pdf"

chunks = partition_pdf(
    filename=file_path,
    infer_table_structure=True,
    strategy="hi_res",
    extract_image_block_types=["Image"],
    extract_image_block_to_payload=True,
    chunking_strategy="by_title",
    max_characters=10000,
    combine_text_under_n_chars=2000,
    new_after_n_chars=6000,
)

In [ ]:
set([str(type(el)) for el in chunks])

In [ ]:
chunks[3].metadata.orig_elements

In [ ]:
elements = chunks[3].metadata.orig_elements
chunk_images = [el for el in elements if 'Image' in str(type(el))]
chunk_images[0].to_dict()

seprate extracted elements into tables,text and images

In [ ]:
tables = []
texts = []

for chunk in chunks:
    if "Table" in str(type(chunk)):
        tables.append(chunk)
    if "CompositeElement" in str(type(chunk)):
        texts.append(chunk)

In [ ]:
#images from the compositeElements objects
def get_images_base64(chunks):
    images_b64 = []
    for chunk in chunks:
        if "CompositeElement" in str(type(chunk)):
            chunk_els = chunk.metadata.orig_elements
            for el in chunk_els:
                if "Image" in str(type(el)):
                    images_b64.append(el.metadata.image_base64)
    return images_b64
images = get_images_base64(chunks)

In [ ]:
import base64
from IPython.display import display, Image

def display_base64_image(base64_code):
    image_data = base64.b64decode(base64_code)
    display(Image(data=image_data))
display_base64_image(images[0])

create a summary of each element extracted from pdf

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

prompt_text = """
You are an assistant tasked with summarizing tables and text.
Give a concise summary of the table or text.

Respond only with the summary.
{text}
"""

prompt = ChatPromptTemplate.from_template(prompt_text)

model = ChatOllama(model="phi3:mini")

summarize_chain = (
    {"text": lambda x: x}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
text_summaries = summarize_chain.batch(
    texts,
    {"max_concurrency": 2}
)

tables_html = [
    table.metadata.text_as_html
    for table in tables
]

table_summaries = summarize_chain.batch(
    tables_html,
    {"max_concurrency": 2}
)

In [ ]:
text_summaries

In [ ]:
import ollama

image_summaries = []

for img in images:
    response = ollama.chat(
        model="llava",
        messages=[
            {
                "role": "user",
                "content": """
Describe this image in detail.
The image comes from a research paper about transformers.
Explain diagrams, plots, tables, arrows, and architecture details.
""",
                "images": [img]
            }
        ]
    )

    image_summaries.append(
        response["message"]["content"]
    )

In [ ]:
image_summaries

In [ ]:
print(image_summaries[1])

creating vectorstore

In [ ]:
import uuid

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_ollama import OllamaEmbeddings

# Vector store
vectorstore = Chroma(
    collection_name="multi_modal_rag",
    embedding_function=OllamaEmbeddings(
        model="nomic-embed-text"
    )
)

# Parent document store
store = InMemoryStore()

id_key = "doc_id"

# Retriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)